In [0]:
%pip install sentence-transformers pandas langchain langchain-community langchain-text-splitters
dbutils.library.restartPython()

In [0]:
import pandas as pd
import os
from sentence_transformers import SentenceTransformer

# 1. Provide your Hugging Face Token
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "enter hugging face"

# 2. Load the Datasets from your workspace.default catalog
print("Loading Schemes datasets...")
try:
    # Reading CSV files from volume using spark.read.csv()
    schemes_df = spark.read.csv("/Volumes/workspace/defaut/schemes_dataset/schemes.csv", header=True, inferSchema=True).toPandas()
    faqs_df = spark.read.csv("/Volumes/workspace/default/schemes_dataset/schemes_faqs.csv", header=True, inferSchema=True).toPandas()
    print(f"Successfully loaded {len(schemes_df)} schemes and {len(faqs_df)} FAQs.")
except Exception as e:
    print("ERROR: Could not find the tables. Did you upload them as 'schemes_dataset' and 'schemes_faqs' in workspace.default?")
    raise e

# 3. Concatenate Text for Vectorization (Per your instructions)
print("Preparing text for embedding...")
# Replace any empty fields with blank spaces so the code doesn't crash
schemes_df.fillna('', inplace=True)

# We merge the columns into one big descriptive text block for the AI to read
schemes_df['Combined_Text'] = (
    "Scheme Name: " + schemes_df['scheme_name'].astype(str) + "\n" +
    "Description: " + schemes_df['detailed_description'].astype(str) + "\n" +
    "Eligibility: " + schemes_df['eligibility'].astype(str)
)

# 4. Initialize our "Made in India" Embedding Model
print("Loading L3Cube Pune Indic-Sentence-BERT model...")
model = SentenceTransformer('l3cube-pune/indic-sentence-bert-nli')

# 5. Generate Vector Embeddings ONLY for the main schemes
print("Generating mathematical vectors for all schemes (This might take a minute)...")
schemes_df['Embedding_Vector'] = schemes_df['Combined_Text'].apply(lambda x: model.encode(str(x)).tolist())

# 6. Save BOTH tables to Databricks Delta Lake
print("Saving Schemes Vector Table...")
spark_schemes = spark.createDataFrame(schemes_df)
spark_schemes.write.format("delta").mode("overwrite").saveAsTable("workspace.default.schemes_vector_table")

print("Saving FAQs Table (Un-embedded, for exact matching later)...")
spark_faqs = spark.createDataFrame(faqs_df)
spark_faqs.write.format("delta").mode("overwrite").saveAsTable("workspace.default.schemes_faqs_table")

print("SUCCESS! Both Schemes and FAQs are ready in workspace.default.")